# Team Season - team_player_onoff_details

This notebook inspects the data **downloaded** from the `team_player_onoff_details` endpoint.

Source folder: `02_data/01_raw/2025_26/02_team_season/team_player_onoff_details`

Scope:
- Read parquet files in the folder
- Show how many files were downloaded and how many rows/columns they contain
- Report missing values (null cells) overall and by row/column distribution

Notes:
- Read-only analysis: no exports or writes


In [25]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve project root by walking up to find 02_data
project_root = Path.cwd()
for _ in range(6):
    if (project_root / "02_data").exists():
        break
    project_root = project_root.parent

endpoint = "team_player_onoff_details"
data_dir = project_root / "02_data" / "01_raw" / "2025_26" / "02_team_season" / endpoint

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)


In [26]:
files = sorted(data_dir.glob("*.parquet"))
print(f"Endpoint: {endpoint}")
print(f"Folder: {data_dir}")
print(f"Parquet files: {len(files)}")

files_df = pd.DataFrame({
    "file": [f.name for f in files],
    "size_mb": [round(f.stat().st_size / 1e6, 3) for f in files],
})
files_df


Endpoint: team_player_onoff_details
Folder: /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/01_raw/2025_26/02_team_season/team_player_onoff_details
Parquet files: 30


,file,size_mb
0,team_player_onoff_details__team_id=1610612737....,0.047
1,team_player_onoff_details__team_id=1610612738....,0.044
2,team_player_onoff_details__team_id=1610612739....,0.046
3,team_player_onoff_details__team_id=1610612740....,0.044
4,team_player_onoff_details__team_id=1610612741....,0.048
5,team_player_onoff_details__team_id=1610612742....,0.046
6,team_player_onoff_details__team_id=1610612743....,0.044
7,team_player_onoff_details__team_id=1610612744....,0.045
8,team_player_onoff_details__team_id=1610612745....,0.044
9,team_player_onoff_details__team_id=1610612746....,0.046


In [27]:
dfs = [pd.read_parquet(f) for f in files]

per_file = pd.DataFrame({
    "file": [f.name for f in files],
    "rows": [len(df) for df in dfs],
    "cols": [df.shape[1] for df in dfs],
})
per_file


,file,rows,cols
0,team_player_onoff_details__team_id=1610612737....,51,64
1,team_player_onoff_details__team_id=1610612738....,40,64
2,team_player_onoff_details__team_id=1610612739....,46,64
3,team_player_onoff_details__team_id=1610612740....,38,64
4,team_player_onoff_details__team_id=1610612741....,57,64
5,team_player_onoff_details__team_id=1610612742....,45,64
6,team_player_onoff_details__team_id=1610612743....,36,64
7,team_player_onoff_details__team_id=1610612744....,42,64
8,team_player_onoff_details__team_id=1610612745....,35,64
9,team_player_onoff_details__team_id=1610612746....,50,64


In [28]:
df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print(f"Combined shape: {df.shape}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {df.shape[1] if len(df.columns) else 0}")


Combined shape: (1297, 64)
Total rows: 1297
Total columns: 64


In [29]:
rows, cols = df.shape
total_cells = rows * cols
null_cells = int(df.isna().sum().sum()) if total_cells else 0
null_pct = (null_cells / total_cells) if total_cells else 0

summary = pd.DataFrame({
    "rows": [rows],
    "cols": [cols],
    "total_cells": [total_cells],
    "null_cells": [null_cells],
    "null_pct": [round(null_pct * 100, 3)],
})
summary


,rows,cols,total_cells,null_cells,null_pct
0,1297,64,83008,1357,1.635


In [30]:
if df.empty:
    col_summary = pd.DataFrame()
else:
    col_nulls = df.isna().sum()
    col_null_pct = df.isna().mean()
    col_summary = (
        pd.DataFrame({
            "null_cells": col_nulls,
            "null_pct": (col_null_pct * 100).round(3),
        })
        .sort_values("null_pct", ascending=False)
    )

col_summary


,null_cells,null_pct
GROUP_VALUE,1267,97.687
COURT_STATUS,30,2.313
VS_PLAYER_NAME,30,2.313
VS_PLAYER_ID,30,2.313
GROUP_SET,0,0.000
FG3A_RANK,0,0.000
FT_PCT_RANK,0,0.000
FTA_RANK,0,0.000
FTM_RANK,0,0.000
FG3_PCT_RANK,0,0.000


In [31]:
if df.empty:
    row_dist = pd.DataFrame()
else:
    row_null_pct = df.isna().mean(axis=1)
    bins = [0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
    row_bins = pd.cut(row_null_pct, bins=bins, include_lowest=True)
    row_counts = row_bins.value_counts().sort_index()
    row_dist = pd.DataFrame({
        "rows": row_counts,
        "pct_rows": (row_counts / len(row_null_pct) * 100).round(3),
    })

row_dist


,rows,pct_rows
"(-0.001, 0.01]",0,0.0
"(0.01, 0.05]",1297,100.0
"(0.05, 0.1]",0,0.0
"(0.1, 0.25]",0,0.0
"(0.25, 0.5]",0,0.0
"(0.5, 0.75]",0,0.0
"(0.75, 1.0]",0,0.0


In [32]:
# Merge details with summary to fill missing columns (no row increase), then export

files = sorted(data_dir.glob("*.parquet"))
dfs = [pd.read_parquet(f) for f in files]
df_details = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

summary_dir = project_root / "02_data" / "01_raw" / "2025_26" / "02_team_season" / "team_player_onoff_summary"
summary_files = sorted(summary_dir.glob("*.parquet"))
summary_dfs = [pd.read_parquet(f) for f in summary_files]
df_summary = pd.concat(summary_dfs, ignore_index=True) if summary_dfs else pd.DataFrame()

if df_details.empty:
    print("No details data to merge.")
else:
    # Use VS_PLAYER_ID instead of PLAYER_ID if present
    details_key_map = {
        "TEAM_ID": "TEAM_ID",
        "PLAYER_ID": "VS_PLAYER_ID" if "VS_PLAYER_ID" in df_details.columns else "PLAYER_ID",
        "COURT_STATUS": "COURT_STATUS",
    }
    summary_key_map = {
        "TEAM_ID": "TEAM_ID",
        "PLAYER_ID": "VS_PLAYER_ID" if "VS_PLAYER_ID" in df_summary.columns else "PLAYER_ID",
        "COURT_STATUS": "COURT_STATUS",
    }

    # Build actual key lists
    details_keys = [details_key_map[k] for k in ["TEAM_ID", "PLAYER_ID", "COURT_STATUS"]]
    summary_keys = [summary_key_map[k] for k in ["TEAM_ID", "PLAYER_ID", "COURT_STATUS"]]

    missing_details = [k for k in details_keys if k not in df_details.columns]
    missing_summary = [k for k in summary_keys if k not in df_summary.columns]

    if df_summary.empty:
        print("No summary data found; exporting details only.")
        df_out = df_details.copy()
    elif missing_details or missing_summary:
        print(f"Missing merge keys in details: {missing_details}")
        print(f"Missing merge keys in summary: {missing_summary}")
        df_out = df_details.copy()
    else:
        before_rows = len(df_details)

        # Rename keys to align for merge
        df_details_m = df_details.rename(columns={details_keys[0]: "TEAM_ID", details_keys[1]: "PLAYER_ID", details_keys[2]: "COURT_STATUS"})
        df_summary_m = df_summary.rename(columns={summary_keys[0]: "TEAM_ID", summary_keys[1]: "PLAYER_ID", summary_keys[2]: "COURT_STATUS"})

        # Merge summary onto details without increasing rows
        df_merged = df_details_m.merge(
            df_summary_m,
            on=["TEAM_ID", "PLAYER_ID", "COURT_STATUS"],
            how="left",
            suffixes=("", "_SUMMARY"),
        )

        # For any columns that exist in summary only, append them.
        # For overlapping columns, fill nulls in details from summary.
        summary_cols = [c for c in df_summary_m.columns if c not in ["TEAM_ID", "PLAYER_ID", "COURT_STATUS"]]
        for c in summary_cols:
            c_summary = f"{c}_SUMMARY"
            if c in df_details_m.columns:
                if c_summary in df_merged.columns:
                    df_merged[c] = df_merged[c].fillna(df_merged[c_summary])
            else:
                if c_summary in df_merged.columns:
                    df_merged[c] = df_merged[c_summary]

        # Drop helper summary columns
        drop_cols = [c for c in df_merged.columns if c.endswith("_SUMMARY")]
        if drop_cols:
            df_merged = df_merged.drop(columns=drop_cols)

        after_rows = len(df_merged)
        if after_rows != before_rows:
            print(f"Warning: row count changed from {before_rows} to {after_rows}")
        else:
            print(f"Rows preserved: {after_rows}")

        df_out = df_merged

    out_dir = project_root / "02_data" / "02_cleaned" / "2025_26" / "02_team_season"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "team_player_onoff_details.parquet"
    df_out.to_parquet(out_path, index=False)
    print(f"Saved to: {out_path}")


Rows preserved: 1297
Saved to: /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_player_onoff_details.parquet
